In [1]:
import pandas as pd
import numpy as np
import random

print("Python environment is working!")

Python environment is working!


In [2]:
random.seed(42)
np.random.seed(42)

print("Random seed configured.")

Random seed configured.


In [3]:
accounts = pd.read_csv("../data/raw/DimAccount.csv")

print("Accounts loaded:", len(accounts))

Accounts loaded: 75000


In [4]:
accounts.head()

,AccountID,CustomerID,AccountNumber,AccountType,BranchID,OpenDate,CurrentBalance,Status
0,1,47966,ACCT-00000001,Business Checking,74,2021-09-07,29843.19,Active
1,2,19734,ACCT-00000002,Business Savings,88,2026-01-31,1455.29,Active
2,3,9970,ACCT-00000003,Checking,55,2021-02-07,3128.56,Active
3,4,35970,ACCT-00000004,Checking,43,2024-06-17,2409.06,Active
4,5,16734,ACCT-00000005,Money Market,6,2023-07-04,5635.30,Active


In [5]:
num_transactions = 1_000_000

print(f"Transactions to generate: {num_transactions:,}")

Transactions to generate: 1,000,000


In [6]:
active_accounts = accounts[
    accounts["Status"] == "Active"
].copy()

print("Active accounts:", len(active_accounts))

Active accounts: 67562


In [7]:
transaction_account_ids = np.random.choice(
    active_accounts["AccountID"].values,
    size=num_transactions,
    replace=True
)

In [8]:
account_lookup = accounts[
    [
        "AccountID",
        "CustomerID",
        "BranchID",
        "AccountType"
    ]
].drop_duplicates("AccountID")

In [9]:
transactions = pd.DataFrame({
    "AccountID": transaction_account_ids
})

transactions = transactions.merge(
    account_lookup,
    on="AccountID",
    how="left"
)

transactions.head()

,AccountID,CustomerID,BranchID,AccountType
0,17499,11028,84,Checking
1,949,38898,99,Savings
2,60880,5887,73,Checking
3,6908,387,2,Savings
4,41219,7765,37,Savings


In [10]:
transaction_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range(
            start="2020-01-01",
            end="2026-07-31",
            freq="D"
        ),
        size=num_transactions,
        replace=True
    )
)

transactions["TransactionDate"] = transaction_dates

In [11]:
transactions["DateKey"] = (
    transactions["TransactionDate"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

In [12]:
transaction_types = np.random.choice(
    [
        "Deposit",
        "Withdrawal",
        "Transfer",
        "Bill Payment",
        "Card Purchase",
        "ATM Withdrawal",
        "Loan Payment"
    ],
    size=num_transactions,
    p=[
        0.15,
        0.10,
        0.20,
        0.12,
        0.30,
        0.08,
        0.05
    ]
)

transactions["TransactionType"] = transaction_types

In [13]:
channels = np.random.choice(
    [
        "Mobile",
        "Online",
        "Branch",
        "ATM"
    ],
    size=num_transactions,
    p=[
        0.45,
        0.30,
        0.15,
        0.10
    ]
)

transactions["Channel"] = channels

In [14]:
amounts = np.random.lognormal(
    mean=5.8,
    sigma=1.15,
    size=num_transactions
)

In [15]:
amounts = np.clip(
    amounts,
    5,
    50_000
)

In [16]:
amount_multiplier = np.select(
    [
        transactions["TransactionType"].eq("Deposit"),
        transactions["TransactionType"].eq("Withdrawal"),
        transactions["TransactionType"].eq("Transfer"),
        transactions["TransactionType"].eq("Bill Payment"),
        transactions["TransactionType"].eq("Card Purchase"),
        transactions["TransactionType"].eq("ATM Withdrawal"),
        transactions["TransactionType"].eq("Loan Payment")
    ],
    [
        3.0,
        1.5,
        4.0,
        1.0,
        0.7,
        1.2,
        2.0
    ],
    default=1.0
)

transactions["Amount"] = np.round(
    amounts * amount_multiplier,
    2
)

In [17]:
transactions["TransactionStatus"] = np.random.choice(
    [
        "Success",
        "Failed",
        "Pending"
    ],
    size=num_transactions,
    p=[
        0.965,
        0.025,
        0.010
    ]
)

In [18]:
base_processing_time = np.random.normal(
    loc=12,
    scale=5,
    size=num_transactions
)

base_processing_time = np.clip(
    base_processing_time,
    2,
    60
)

In [19]:
channel_multiplier = transactions["Channel"].map({
    "Mobile": 0.70,
    "Online": 0.80,
    "Branch": 1.80,
    "ATM": 1.20
})

In [20]:
transactions["ProcessingTimeSeconds"] = np.round(
    base_processing_time * channel_multiplier
).astype(int)

transactions["ProcessingTimeSeconds"] = np.clip(
    transactions["ProcessingTimeSeconds"],
    1,
    120
)

In [21]:
fee_rate = transactions["TransactionType"].map({
    "Deposit": 0.000,
    "Withdrawal": 0.001,
    "Transfer": 0.002,
    "Bill Payment": 0.001,
    "Card Purchase": 0.010,
    "ATM Withdrawal": 0.005,
    "Loan Payment": 0.000
})

In [22]:
transactions["FeeAmount"] = np.round(
    transactions["Amount"] * fee_rate,
    2
)

In [23]:
transactions["RevenueAmount"] = transactions["FeeAmount"]

In [24]:
fact_transactions = transactions[
    [
        "CustomerID",
        "AccountID",
        "BranchID",
        "DateKey",
        "TransactionType",
        "Amount",
        "ProcessingTimeSeconds",
        "FeeAmount",
        "RevenueAmount",
        "TransactionStatus",
        "Channel"
    ]
].copy()

In [25]:
fact_transactions.head()

,CustomerID,AccountID,BranchID,DateKey,TransactionType,Amount,ProcessingTimeSeconds,FeeAmount,RevenueAmount,TransactionStatus,Channel
0,11028,17499,84,20201010,Card Purchase,50.58,10,0.51,0.51,Success,Mobile
1,38898,949,99,20241104,Withdrawal,6806.92,10,6.81,6.81,Success,Mobile
2,5887,60880,73,20200223,Card Purchase,306.36,14,3.06,3.06,Success,Mobile
3,387,6908,2,20210317,Loan Payment,85.44,11,0.00,0.00,Success,ATM
4,7765,41219,37,20260725,Loan Payment,1497.11,7,0.00,0.00,Success,Mobile


In [26]:
print(fact_transactions.shape)

(1000000, 11)


In [27]:
fact_transactions.isnull().sum()

CustomerID               0
AccountID                0
BranchID                 0
DateKey                  0
TransactionType          0
Amount                   0
ProcessingTimeSeconds    0
FeeAmount                0
RevenueAmount            0
TransactionStatus        0
Channel                  0
dtype: int64

In [28]:
(fact_transactions["Amount"] < 0).sum()

np.int64(0)

In [29]:
fact_transactions["CustomerID"].min(), fact_transactions["CustomerID"].max()

(1, 50000)

In [30]:
fact_transactions["AccountID"].min(), fact_transactions["AccountID"].max()

(1, 75000)

In [31]:
fact_transactions["BranchID"].min(), fact_transactions["BranchID"].max()

(1, 100)

In [32]:
fact_transactions["DateKey"].min(), fact_transactions["DateKey"].max()

(20200101, 20260731)

In [33]:
fact_transactions["TransactionType"].value_counts()

TransactionType
Card Purchase     300817
Transfer          200377
Deposit           149919
Bill Payment      119412
Withdrawal         99680
ATM Withdrawal     80017
Loan Payment       49778
Name: count, dtype: int64

In [34]:
fact_transactions["Channel"].value_counts()

Channel
Mobile    449715
Online    300280
Branch    150354
ATM        99651
Name: count, dtype: int64

In [35]:
fact_transactions["Amount"].mean()

np.float64(1232.81175819)

In [36]:
fact_transactions["ProcessingTimeSeconds"].mean()

np.float64(11.37885)

In [37]:
fact_transactions["ProcessingTimeSeconds"].mean()

np.float64(11.37885)

In [38]:
valid_account_ids = set(accounts["AccountID"])

invalid_accounts = (
    ~fact_transactions["AccountID"].isin(valid_account_ids)
).sum()

print("Invalid AccountIDs:", invalid_accounts)

Invalid AccountIDs: 0


In [39]:
account_relationships = accounts[
    [
        "AccountID",
        "CustomerID",
        "BranchID"
    ]
].drop_duplicates("AccountID")

In [40]:
validation = fact_transactions.merge(
    account_relationships,
    on="AccountID",
    how="left",
    suffixes=(
        "_Transaction",
        "_Account"
    )
)

In [41]:
customer_mismatches = (
    validation["CustomerID_Transaction"]
    != validation["CustomerID_Account"]
).sum()

print("Customer mismatches:", customer_mismatches)

Customer mismatches: 0


In [42]:
branch_mismatches = (
    validation["BranchID_Transaction"]
    != validation["BranchID_Account"]
).sum()

print("Branch mismatches:", branch_mismatches)

Branch mismatches: 0


In [43]:
transaction_output_path = "../data/raw/FactTransactions.csv"

fact_transactions.to_csv(
    transaction_output_path,
    index=False
)

print(
    f"Saved {len(fact_transactions):,} transactions to "
    f"{transaction_output_path}"
)

Saved 1,000,000 transactions to ../data/raw/FactTransactions.csv


In [44]:
python_total_transactions = len(fact_transactions)

python_total_value = fact_transactions["Amount"].sum()

python_total_revenue = fact_transactions["RevenueAmount"].sum()

python_avg_processing = fact_transactions[
    "ProcessingTimeSeconds"
].mean()

print("Python results")
print("-----------------------------")
print(f"Transactions: {python_total_transactions:,}")
print(f"Transaction value: ${python_total_value:,.2f}")
print(f"Revenue: ${python_total_revenue:,.2f}")
print(f"Avg processing time: {python_avg_processing:.2f} seconds")

Python results
-----------------------------
Transactions: 1,000,000
Transaction value: $1,232,811,758.19
Revenue: $2,854,552.48
Avg processing time: 11.38 seconds
